In [0]:
display(
    dbutils.fs.ls("/Volumes/workspace/default/dsp_pipeline")
)

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/dsp_pipeline/advertisers.csv,advertisers.csv,401,1784447296000
dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,events_day1.csv,3094332,1784447300000
dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,events_day2.csv,4048441,1784447301000


## Bronze Layer – Raw Data Ingestion

Ingest raw vendor files as-is, with no cleaning or deduplication. 

Only metadata columns (`pipeline_ingest_time`, `source_file`) are added for lineage. 

v1/v2 schema drift is preserved by keeping the two event files as separate bronze tables — reconciliation happens explicitly in Silver, not silently here.

In [0]:
raw_path = "/Volumes/workspace/default/dsp_pipeline"
bronze_path = "/Volumes/workspace/default/dsp_pipeline/bronze"

print(raw_path)
print(bronze_path)

/Volumes/workspace/default/dsp_pipeline
/Volumes/workspace/default/dsp_pipeline/bronze


In [0]:
from pyspark.sql.functions import col, current_timestamp

def read_raw(filename):
    """Read a vendor CSV with header/schema inference, keeping source file path for lineage."""
    return (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(f"{raw_path}/{filename}")
        .select("*", "_metadata")
    )

events_day1_raw = read_raw("events_day1.csv")
events_day2_raw = read_raw("events_day2.csv")
advertisers_raw = read_raw("advertisers.csv")

### Schema drift: v1 (day1) vs v2 (day2)

- `spend` (v1) is renamed to `media_cost` (v2) — same meaning, different column name.

- `viewability` is new in v2, not present in v1.

- All other columns (`event_id`, `event_type`, `event_time`, `ingest_time`, 
  `advertiser_id`, `campaign_id`, `placement`, `device`, `geo`, `currency`) are unchanged.

Bronze keeps day1 and day2 as separate tables to preserve raw fidelity exactly as the 
vendor sent it. 

The rename/reconciliation (`media_cost` → `spend`, filling `viewability` 
as null for v1 rows) happens in Silver, where the unified schema is defined.

In [0]:
print("=== v1 (day1) schema ===")
events_day1_raw.printSchema()

print("=== v2 (day2) schema ===")
events_day2_raw.printSchema()

print("=== advertisers schema ===")
advertisers_raw.printSchema()

=== v1 (day1) schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingest_time: timestamp (nullable = true)
 |-- advertiser_id: string (nullable = true)
 |-- campaign_id: string (nullable = true)
 |-- placement: string (nullable = true)
 |-- device: string (nullable = true)
 |-- geo: string (nullable = true)
 |-- spend: double (nullable = true)
 |-- currency: string (nullable = true)
 |-- _metadata: struct (nullable = false)
 |    |-- file_path: string (nullable = false)
 |    |-- file_name: string (nullable = false)
 |    |-- file_size: long (nullable = false)
 |    |-- file_block_start: long (nullable = false)
 |    |-- file_block_length: long (nullable = false)
 |    |-- file_modification_time: timestamp (nullable = false)

=== v2 (day2) schema ===
root
 |-- event_id: string (nullable = true)
 |-- event_type: string (nullable = true)
 |-- event_time: string (nullable = true)
 |-- ingest_time:

In [0]:
def add_bronze_metadata(df):
    """Attach ingest timestamp and source file path for auditability — no other changes."""
    return (
        df
        .withColumn("pipeline_ingest_time", current_timestamp())
        .withColumn("source_file", col("_metadata.file_path"))
        .drop("_metadata")
    )

bronze_events_day1 = add_bronze_metadata(events_day1_raw)
bronze_events_day2 = add_bronze_metadata(events_day2_raw)
bronze_advertisers = add_bronze_metadata(advertisers_raw)

In [0]:
bronze_events_day1.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_events_day1")
bronze_events_day2.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_events_day2")
bronze_advertisers.write.format("delta").mode("overwrite").saveAsTable("workspace.default.bronze_advertisers")

In [0]:
# Sanity check only — row counts + samples, not part of the pipeline logic.
for t in ["bronze_events_day1", "bronze_events_day2", "bronze_advertisers"]:
    n = spark.table(f"workspace.default.{t}").count()
    print(f"{t}: {n} rows")

display(spark.table("workspace.default.bronze_events_day1").limit(5))
display(spark.table("workspace.default.bronze_events_day2").limit(5))

bronze_events_day1: 26118 rows
bronze_events_day2: 31498 rows
bronze_advertisers: 10 rows


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file
evt_7e681157,impression,2026-06-01T03:11:56Z,2026-06-01T09:17:54.000Z,adv_1151,cmp_1151_1,banner_300x250,mobile,IN,0.0194,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7d680fc4,impression,2026-06-01T06:32:35Z,2026-06-01T05:33:13.000Z,adv_1186,cmp_1186_2,interstitial,ctv,IN,0.0102,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_8068147d,click,2026-06-01T18:56:56Z,2026-06-01T01:16:09.000Z,adv_1151,cmp_1151_3,native_feed,mobile,GB,0.0212,INR,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7f6812ea,impression,2026-06-01T20:16:21Z,2026-06-01T03:29:28.000Z,adv_1172,cmp_1172_2,preroll,ctv,DE,0.018,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv
evt_7a680b0b,impression,2026-06-01T07:43:12Z,2026-06-01T08:48:22.000Z,adv_1165,cmp_1165_1,banner_728x90,desktop,GB,0.0189,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,media_cost,currency,viewability,pipeline_ingest_time,source_file
evt_a815d507,impression,2026-06-04 18:50:42+05:30,2026-06-04T00:18:44.000Z,adv_1144,cmp_1144_2,interstitial,desktop,US,0.0185,USD,0.68,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a815d507,impression,2026-06-04 18:50:42+05:30,2026-06-04T00:18:44.000Z,adv_1144,cmp_1144_2,interstitial,desktop,US,0.0185,USD,0.68,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a715d374,impression,2026-06-04 20:03:11+05:30,2026-06-04T06:56:34.000Z,adv_1151,cmp_1151_1,interstitial,ctv,DE,0.0138,USD,0.48,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_aa15d82d,impression,2026-06-04 04:57:18+05:30,2026-06-04T19:06:57.000Z,adv_1172,cmp_1172_3,preroll,mobile,unknown,0.0131,USD,0.54,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv
evt_a915d69a,impression,2026-06-04 10:59:30+05:30,2026-06-04T09:08:41.000Z,adv_1137,cmp_1137_2,banner_728x90,mobile,SG,0.0013,USD,0.87,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv


## Silver Layer – Unified, Clean Events

Reconcile v1/v2 schema drift into one events table.

Deduplicate (exact duplicates and 
re-emitted event_ids, latest ingest_time wins).

Normalise all timestamps to UTC, and 
quarantine rows that can't be repaired (malformed timestamps, negative cost) instead 
of silently dropping them.

In [0]:
bronze_events_day1 = spark.table("workspace.default.bronze_events_day1")
bronze_events_day2 = spark.table("workspace.default.bronze_events_day2")
bronze_advertisers = spark.table("workspace.default.bronze_advertisers")

In [0]:
from pyspark.sql.functions import lit

events_day1_unified = (
    bronze_events_day1
    .withColumn("viewability", lit(None).cast("double"))
    .withColumn("schema_version", lit("v1"))
)

events_day2_unified = (
    bronze_events_day2
    .withColumnRenamed("media_cost", "spend")
    .withColumn("schema_version", lit("v2"))
)

events_unified = events_day1_unified.unionByName(events_day2_unified)

In [0]:
display(
    events_unified
    .select("schema_version", "event_time")
    .distinct()
    .limit(50)
)

schema_version,event_time
v1,2026-06-01T03:11:56Z
v1,2026-06-01T06:32:35Z
v1,2026-06-01T18:56:56Z
v1,2026-06-01T20:16:21Z
v1,2026-06-01T07:43:12Z
v1,2026-06-01T18:51:56Z
v1,2026-06-01T14:26:37Z
v1,2026-06-01T14:35:12Z
v1,2026-06-01T21:29:58Z
v1,2026-06-01T23:42:59Z


In [0]:
from pyspark.sql.functions import (
    col,
    lit,
    current_timestamp,
    to_timestamp,
    when,
    row_number
)

display(
    events_unified
    .filter(events_unified.schema_version == "v2")
    .select("event_time")
    .distinct()
    .limit(20)
)

event_time
2026-06-04 18:50:42+05:30
2026-06-04 20:03:11+05:30
2026-06-04 04:57:18+05:30
2026-06-04 10:59:30+05:30
2026-06-04 15:21:11+05:30
2026-06-04 11:36:19+05:30
2026-06-04 12:01:32+05:30
2026-06-04 17:45:20+05:30
2026-06-04 05:16:59+05:30
2026-06-04 19:29:29+05:30


In [0]:
spark.sql("SET spark.sql.session.timeZone").show(truncate=False)

+--------------------------+-------+
|key                       |value  |
+--------------------------+-------+
|spark.sql.session.timeZone|Etc/UTC|
+--------------------------+-------+



In [0]:
from pyspark.sql.functions import col, coalesce, try_to_timestamp, lit

events_with_ts = (
    events_unified.withColumn(
        "event_time_utc",
        coalesce(
            try_to_timestamp(col("event_time"), lit("yyyy-MM-dd'T'HH:mm:ss'Z'")),
            try_to_timestamp(col("event_time"), lit("yyyy-MM-dd HH:mm:ssXXX"))
        )
    )
)

display(
    events_with_ts
    .filter(col("event_time_utc").isNull())
    .select("schema_version", "event_id", "event_time")
)

schema_version,event_id,event_time
v1,evt_eac9aabb,N/A
v1,evt_efc9b29a,N/A
v1,evt_2efc06ff,N/A
v1,evt_25844331,N/A
v1,evt_2696dbb3,N/A
v1,evt_e04f6eb,N/A
v1,evt_cc709db3,N/A
v1,evt_d372e74f,N/A
v1,evt_7195ef4d,N/A
v1,evt_d32a6cad,N/A


In [0]:
from pyspark.sql.functions import col, when

# Rows are quarantined (not dropped) if:
#   - event_time could not be parsed into a valid UTC timestamp
#   - spend is negative
#
# This preserves bad records for audit and vendor investigation while allowing
# the rest of the pipeline to continue.

quarantine_condition = (
    col("event_time_utc").isNull() |
    (col("spend") < 0)
)

silver_events_quarantine = (
    events_with_ts
    .filter(quarantine_condition)
    .withColumn(
        "quarantine_reason",
        when(
            col("event_time_utc").isNull(),
            "malformed_event_time"
        ).when(
            col("spend") < 0,
            "negative_spend"
        )
    )
)

events_clean = (
    events_with_ts
    .filter(~quarantine_condition)
)

In [0]:
print("Malformed timestamps:",
      events_with_ts.filter(col("event_time_utc").isNull()).count())
print("Negative spend:",
      events_with_ts.filter(col("spend") < 0).count())
print("Null spend:",
      events_with_ts.filter(col("spend").isNull()).count())

Malformed timestamps: 175
Negative spend: 286
Null spend: 0


In [0]:
silver_events_quarantine.count()

461

In [0]:
events_with_ts.filter(quarantine_condition).count()

461

In [0]:
events_with_ts.filter(
    col("event_time_utc").isNull() &
    (col("spend") < 0)
).count()

0

In [0]:
#1 & 2. Dedup (exact duplicates, then latest-ingest_time-wins per event_id):
from pyspark.sql import Window
from pyspark.sql.functions import row_number, desc, col

# Step 1: drop exact duplicate rows (identical across every column)
events_deduped = events_clean.dropDuplicates()

# Step 2: for re-emitted event_ids (same id, different content/ingest_time — a
# correction, not a duplicate), keep only the latest ingest_time per event_id.
# A naive dropDuplicates() would keep an arbitrary row, potentially the stale one.
window_spec = (
    Window
    .partitionBy("event_id")
    .orderBy(
        col("ingest_time").desc(),
        col("pipeline_ingest_time").desc()
    )
)

silver_events = (
    events_deduped
    .withColumn("rn", row_number().over(window_spec))
    .filter(col("rn") == 1)
    .drop("rn")
)

print(f"events_clean:                {events_clean.count()}")
print(f"after exact-dup drop:        {events_deduped.count()}")
print(f"silver_events (final):       {silver_events.count()}")

events_clean:                57155
after exact-dup drop:        55560
silver_events (final):       54460


### Silver Layer Summary

| Processing Step | Rows |
|-----------------|-----:|
|Valid records after quarantine|57,155|
|After removing exact duplicates|55,560|
|Final Silver table|54,460|

**Observations**

- 461 irreparable records were quarantined (175 malformed timestamps, 286 negative spend).

- 1,595 exact duplicate rows were removed.

- 1,100 older event versions were discarded, keeping only the latest `ingest_time` per `event_id`.

In [0]:
from pyspark.sql.functions import count

dup_check = (
    silver_events.groupBy("event_id")
    .agg(count("*").alias("cnt"))
    .filter(col("cnt") > 1)
)

print(f"Duplicate event_ids remaining in silver_events: {dup_check.count()}")

Duplicate event_ids remaining in silver_events: 0


In [0]:
silver_events.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_events")

silver_events_quarantine.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.silver_events_quarantine")

In [0]:
silver_count = spark.table("workspace.default.silver_events").count()
quarantine_count = spark.table("workspace.default.silver_events_quarantine").count()

print(f"Silver Events: {silver_count}")
print(f"Silver Quarantine: {quarantine_count}")

display(spark.table("workspace.default.silver_events").limit(5))
display(spark.table("workspace.default.silver_events_quarantine").limit(5))

Silver Events: 54460
Silver Quarantine: 461


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file,viewability,schema_version,event_time_utc
evt_100426d6,impression,2026-06-06 11:38:30+05:30,2026-06-06T07:09:09.000Z,adv_1109,cmp_1109_3,banner_728x90,ctv,IN,0.0193,INR,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.91,v2,2026-06-06T06:08:30.000Z
evt_1004fa11,impression,2026-06-01T14:25:52Z,2026-06-01T15:45:11.000Z,adv_1130,cmp_1130_3,banner_728x90,tablet,unknown,0.006,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-01T14:25:52.000Z
evt_1006546,impression,2026-06-01T06:19:43Z,2026-06-01T21:55:26.000Z,adv_1165,cmp_1165_2,banner_728x90,desktop,IN,0.0054,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-01T06:19:43.000Z
evt_1006630d,impression,2026-06-05 03:33:41+05:30,2026-06-05T11:45:02.000Z,adv_1179,cmp_1179_2,banner_728x90,desktop,DE,0.0027,INR,2026-07-19T11:58:35.324Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day2.csv,0.47,v2,2026-06-04T22:03:41.000Z
evt_1007f274,impression,2026-06-03T12:46:37Z,2026-06-03T03:06:49.000Z,adv_1158,cmp_1158_3,interstitial,tablet,DE,0.0054,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-03T12:46:37.000Z


event_id,event_type,event_time,ingest_time,advertiser_id,campaign_id,placement,device,geo,spend,currency,pipeline_ingest_time,source_file,viewability,schema_version,event_time_utc,quarantine_reason
evt_eac9aabb,impression,N/A,2026-06-01T22:01:33.000Z,adv_1109,cmp_1109_2,banner_728x90,mobile,SG,0.0139,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,null,malformed_event_time
evt_efc9b29a,impression,N/A,2026-06-01T17:54:54.000Z,adv_1123,cmp_1123_3,preroll,mobile,SG,0.019,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,null,malformed_event_time
evt_2efc06ff,impression,N/A,2026-06-01T05:09:25.000Z,adv_1144,cmp_1144_3,interstitial,desktop,null,0.0024,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,null,malformed_event_time
evt_445d3b0b,impression,2026-06-01T07:47:46Z,2026-06-01T21:35:15.000Z,adv_1130,cmp_1130_2,native_feed,tablet,GB,-0.0087,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-01T07:47:46.000Z,negative_spend
evt_247775ab,impression,2026-06-01T01:04:43Z,2026-06-01T03:16:46.000Z,adv_1116,cmp_1116_3,native_feed,ctv,null,-0.0181,USD,2026-07-19T11:58:32.366Z,dbfs:/Volumes/workspace/default/dsp_pipeline/events_day1.csv,null,v1,2026-06-01T01:04:43.000Z,negative_spend
